# Glass Bottle LCA

This notebook works out a simplified cradle-to-grave carbon footprint for a 0.5 L glass soft drink bottle. It's built in Brightway and uses only the `biosphere3` database, no BAFU, no ecoinvent, so unlike the PET bottle model it isn't linked to a real background supply chain. Instead, each life cycle stage gets its emissions entered by hand from literature figures, and biosphere3 is used just to turn those numbers into a GWP score. Once the database is written here, it also shows up in Activity Browser, so from this point on I switch between the two: the notebook for building and adjusting the model, Activity Browser for looking at the exchange table, running the calculation setup, and doing the Monte Carlo and contribution analysis visually.

This kind of shortcut is usually called a screening LCA. It makes sense when there's no licensed background database to link to, when the goal is a quick comparison rather than a full study, or when only one impact category (here, climate change) actually matters for the question being asked.

It comes with real limitations, worth stating plainly in any writeup:

- Only what gets typed in counts. A linked model like the PET one pulls in an entire upstream chain automatically; here, anything left out (say, the packaging around the packaging, or capital equipment) is just absent from the result.
- The emission factors below are typical literature figures, not measurements from an actual glass plant. Before this feeds into a real report they should be swapped for numbers from an EPD, ecoinvent, or a national statistics source.
- Only the CO2 flows that drive GWP are entered. A proper LCA would also carry acidification, eutrophication, and the rest, which this approach simply can't produce.

Same reason as before for using a notebook rather than a plain script: getting the numbers right takes some back and forth, and a notebook lets each stage be checked and tweaked on its own without rerunning everything.

What's in here:
1. Project setup
2. Emission factors for each stage, with where each number comes from
3. Building the foreground activity from those factors
4. Running the LCA
5. Contribution breakdown by stage
6. Monte Carlo

## 1. Project setup

This uses its own Brightway project, separate from the PET bottle one, so the two models can't interfere with each other. `bw2setup()` installs `biosphere3` and the standard LCIA methods (IPCC 2013 included), which is all this notebook needs since there's no background database to import here.

In [1]:
import brightway2 as bw

bw.projects.set_current("Glass Bottle LCA")
bw.bw2setup()  # installs biosphere3 and LCIA methods, skip if already run for this project

print(list(bw.databases))

Biosphere database already present!!! No setup is needed
['biosphere3', 'Glass Bottle']


In [2]:
import brightway2 as bw

bw.projects.set_current("Glass Bottle LCA")

act = bw.get_activity(("Glass Bottle", "glass_bottle_0.5L"))
act["reference product"] = act["name"]
act.save()

print(act["reference product"])

Glass bottle, 0.5 litre soft drink, cradle-to-grave


## 2. Emission factors for each life cycle stage

I'm keeping every assumption as a named variable with a comment on where the number comes from, before building anything, so the model stays auditable. Anyone reading this, including my professor, can see exactly what was assumed and change one number to test a different scenario, instead of digging a value out of a longer expression.

The numbers below are typical figures reported in glass packaging LCA literature for container glass in Europe. Swap them for values from a specific source before using this for anything real.

In [3]:
# ---- Physical parameters of the bottle ----
BOTTLE_MASS_KG = 0.250   # a typical single-use 0.5 L glass soft drink bottle weighs about 250 g
DIST_KM = 100             # factory to retailer distance, same assumption as the PET notebook

# ---- End-of-life split ----
# Switzerland has a high glass collection rate; remainder assumed landfilled
# (glass is inert and does not generate emissions in landfill, so it is not
# modelled as a separate emission source here)
EoL_RECYCLE = 0.90
EoL_LANDFILL = 0.10

# ---- Emission factors, in kg CO2-eq per kg of glass unless noted ----
# Raw material extraction and batch preparation (sand, soda ash, limestone quarrying and transport)
EF_RAW_MATERIAL = 0.15

# Melting and forming: furnace energy (natural gas) plus electricity for forming,
# assuming roughly 50% recycled cullet content in the batch
EF_MELT_FORM = 0.55

# Road freight transport, kg CO2-eq per tonne-kilometer (typical diesel lorry, fleet average)
EF_TRUCK_PER_TKM = 0.062

# Recycling: crushing, cleaning, and re-melting preparation of collected cullet
# (credit for avoided virgin material is not applied here, to keep the model simple)
EF_RECYCLING_PROCESSING = 0.02

print("Emission factors set.")

Emission factors set.


## 3. Build the foreground activity

In the PET notebook, every exchange pointed at another activity (a technosphere link), and Brightway pulled in that activity's own emissions automatically. There's no such activity to point at here, since no background database is loaded. So instead, each stage's total CO2-eq gets entered directly as an exchange against a `biosphere3` flow (`'Carbon dioxide, fossil'`), already in the right unit and already multiplied out by mass, distance, or share.

I'm using `'Carbon dioxide, fossil'` specifically because the IPCC GWP100 characterisation factor for it is exactly 1, so entering a kg CO2-eq value directly as kg of this flow gives the right score without needing a separate conversion. It's a common shortcut in screening LCAs where one pollutant dominates the impact category being looked at.

In [4]:
# Find the biosphere3 flow we will use for every stage
co2_candidates = [f for f in bw.Database("biosphere3")
                  if f["name"] == "Carbon dioxide, fossil" and f["categories"] == ("air",)]

co2_flow = co2_candidates[0]
print(co2_flow)

'Carbon dioxide, fossil' (kilogram, None, ('air',))


In [5]:
# Compute each stage's emissions in kg CO2-eq for one bottle
raw_material_kgco2 = BOTTLE_MASS_KG * EF_RAW_MATERIAL
melt_form_kgco2 = BOTTLE_MASS_KG * EF_MELT_FORM
transport_kgco2 = (BOTTLE_MASS_KG * DIST_KM / 1000) * EF_TRUCK_PER_TKM
recycling_kgco2 = BOTTLE_MASS_KG * EoL_RECYCLE * EF_RECYCLING_PROCESSING
# Landfill share contributes no emissions in this simplified model (glass is inert)

print(f"Raw material:      {raw_material_kgco2:.4f} kg CO2-eq")
print(f"Melting/forming:    {melt_form_kgco2:.4f} kg CO2-eq")
print(f"Transport:          {transport_kgco2:.4f} kg CO2-eq")
print(f"Recycling:          {recycling_kgco2:.4f} kg CO2-eq")
print(f"Total (sanity check): {raw_material_kgco2 + melt_form_kgco2 + transport_kgco2 + recycling_kgco2:.4f} kg CO2-eq")

Raw material:      0.0375 kg CO2-eq
Melting/forming:    0.1375 kg CO2-eq
Transport:          0.0016 kg CO2-eq
Recycling:          0.0045 kg CO2-eq
Total (sanity check): 0.1811 kg CO2-eq


In [6]:
DB_NAME = "Glass Bottle"

if DB_NAME in bw.databases:
    del bw.databases[DB_NAME]

foreground = bw.Database(DB_NAME)
foreground.write({
    (DB_NAME, "glass_bottle_0.5L"): {
        "name": "Glass bottle, 0.5 litre soft drink, cradle-to-grave",
        "unit": "piece",
        "location": "CH",
        "type": "process",
        "exchanges": [
            {"input": (DB_NAME, "glass_bottle_0.5L"), "amount": 1, "unit": "piece", "type": "production"},
            {"input": co2_flow.key, "amount": raw_material_kgco2, "unit": "kilogram", "type": "biosphere"},
            {"input": co2_flow.key, "amount": melt_form_kgco2, "unit": "kilogram", "type": "biosphere"},
            {"input": co2_flow.key, "amount": transport_kgco2, "unit": "kilogram", "type": "biosphere"},
            {"input": co2_flow.key, "amount": recycling_kgco2, "unit": "kilogram", "type": "biosphere"},
        ]
    }
})

print("Database written:", list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:18:09
  Finished: 08/06/2026 11:18:09
  Total time elapsed: 00:00:00
  CPU %: 98.30
  Memory %: 2.57
Database written: ['biosphere3', 'Glass Bottle']


All four biosphere exchanges above use the same flow (`co2_flow`), just with different amounts. Brightway sums exchanges pointing at the same flow within one activity automatically, so this is the same as writing one exchange with the combined total. I'm keeping them separate on purpose, so each stage's contribution can still be checked individually in Activity Browser's exchange table, or one stage's assumption can be edited without recalculating everything else by hand.

## 4. Run the LCA

I'm using the same method as the PET notebook (`IPCC 2013, climate change, global warming potential (GWP100)`) so the two results can actually be compared. Changing the method for just one of them would make any comparison meaningless.

In [7]:
bottle = bw.get_activity((DB_NAME, "glass_bottle_0.5L"))

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

lca = bw.LCA({bottle: 1}, method)
lca.lci()
lca.lcia()

print(f"GWP 100a: {lca.score:.4f} kg CO2-eq per bottle")

GWP 100a: 0.1811 kg CO2-eq per bottle


## 5. Contribution breakdown by stage

Even in a simplified model like this, knowing which stage dominates tells you where a redesign would actually help. For most glass containers it's melting and forming energy, if a report only states the total score, that insight is lost.

In [8]:
stages = {
    "Raw material": raw_material_kgco2,
    "Melting/forming": melt_form_kgco2,
    "Transport": transport_kgco2,
    "Recycling (EoL)": recycling_kgco2,
}

total = sum(stages.values())
print(f"Total: {total:.4f} kg CO2-eq per bottle\n")
for name, value in stages.items():
    share = 100 * value / total
    print(f"{name:<18} {value:.4f} kg CO2-eq  ({share:5.1f}%)")

Total: 0.1811 kg CO2-eq per bottle

Raw material       0.0375 kg CO2-eq  ( 20.7%)
Melting/forming    0.1375 kg CO2-eq  ( 75.9%)
Transport          0.0016 kg CO2-eq  (  0.9%)
Recycling (EoL)    0.0045 kg CO2-eq  (  2.5%)


## Comparing with the PET bottle

Both notebooks use the same functional unit (one 0.5 L bottle) and the same impact method, so the two `GWP 100a` scores can be compared directly once both have been run. Worth remembering though that the two models aren't built the same way: the PET one uses linked background data, a fuller, more defensible inventory, while this one uses literature emission factors, faster but rougher. A rigorous glass-versus-PET comparison would ideally use the same modelling depth for both, for instance by linking the glass bottle to a background database too, once one's available.

## 6. Monte Carlo uncertainty analysis

The single `GWP 100a` score above is one point estimate, it hides how much the result could vary given real uncertainty in every exchange behind it. Monte Carlo reruns the calculation many times, resampling every exchange from its uncertainty distribution each time, and returns a spread instead of one number. Reporting a mean and confidence interval is more honest than a single figure, especially when comparing products.

Two things to know first: `bw2calc`'s Monte Carlo solver can raise `ValueError: 'scipy.sparse.linalg.cgs' called with invalid atol='legacy'` on newer `scipy` versions, if that happens, run `conda install "scipy<1.12"` in the `ab` environment and restart the kernel. And Monte Carlo only produces a real spread if the exchanges being sampled actually carry uncertainty data, the foreground exchanges I wrote myself here don't have any set by default, so they'll stay fixed across iterations unless I add it explicitly.

In [9]:
method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

mc = bw.MonteCarloLCA({bottle: 1}, method)
n_iterations = 1000
scores = [next(mc) for _ in range(n_iterations)]

import numpy as np

print(f"Iterations: {n_iterations}")
print(f"Mean:   {np.mean(scores):.4f} kg CO2-eq per bottle")
print(f"Median: {np.median(scores):.4f} kg CO2-eq per bottle")
print(f"Std dev: {np.std(scores):.4f}")
print(f"5th to 95th percentile: {np.percentile(scores, 5):.4f} to {np.percentile(scores, 95):.4f} kg CO2-eq")

Iterations: 1000
Mean:   0.1811 kg CO2-eq per bottle
Median: 0.1811 kg CO2-eq per bottle
Std dev: 0.0000
5th to 95th percentile: 0.1811 to 0.1811 kg CO2-eq


In [10]:
print(list(bw.databases))

['biosphere3', 'Glass Bottle']
